# CUSTOMER SEGMENTATION PROJECT

### PROJECT GOAL:
The goal of this project is cluster the customers into different groups
based on their spending behavioural pattern. This will help understand
the taste of each customers and will help the company to target each customer with the right advert, product and service.

In [23]:
# data wrangling libraries
import pandas as pd
import numpy as np

# visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# data modelling and statistical analysis library
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler,LabelEncoder
from sklearn.cluster import k_means,dbscan


In [2]:
data = pd.read_csv('cleaned_marketing.csv')

In [3]:
data.head()

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138,0,0,2012-09-04,58,635,...,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344,1,1,2014-03-08,38,11,...,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613,0,0,2013-08-21,26,426,...,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646,1,0,2014-02-10,26,11,...,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293,1,0,2014-01-19,94,173,...,5,0,0,0,0,0,0,3,11,0


In [4]:
# check the columns
data.columns

Index(['ID', 'Year_Birth', 'Education', 'Marital_Status', 'Income', 'Kidhome',
       'Teenhome', 'Dt_Customer', 'Recency', 'MntWines', 'MntFruits',
       'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts',
       'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases',
       'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth',
       'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1',
       'AcceptedCmp2', 'Complain', 'Z_CostContact', 'Z_Revenue', 'Response'],
      dtype='object')

In [5]:
# drop the unnecesary columns

data.drop(columns= ['ID','Dt_Customer'], inplace=True)

In [6]:
# check the datatypes of each column and null values

data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2240 entries, 0 to 2239
Data columns (total 27 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Year_Birth           2240 non-null   int64 
 1   Education            2240 non-null   object
 2   Marital_Status       2240 non-null   object
 3   Income               2240 non-null   object
 4   Kidhome              2240 non-null   int64 
 5   Teenhome             2240 non-null   int64 
 6   Recency              2240 non-null   int64 
 7   MntWines             2240 non-null   int64 
 8   MntFruits            2240 non-null   int64 
 9   MntMeatProducts      2240 non-null   int64 
 10  MntFishProducts      2240 non-null   int64 
 11  MntSweetProducts     2240 non-null   int64 
 12  MntGoldProds         2240 non-null   int64 
 13  NumDealsPurchases    2240 non-null   int64 
 14  NumWebPurchases      2240 non-null   int64 
 15  NumCatalogPurchases  2240 non-null   int64 
 16  NumSto

In [7]:
# doublecheck the null values 

data.isnull().sum()

Year_Birth             0
Education              0
Marital_Status         0
Income                 0
Kidhome                0
Teenhome               0
Recency                0
MntWines               0
MntFruits              0
MntMeatProducts        0
MntFishProducts        0
MntSweetProducts       0
MntGoldProds           0
NumDealsPurchases      0
NumWebPurchases        0
NumCatalogPurchases    0
NumStorePurchases      0
NumWebVisitsMonth      0
AcceptedCmp3           0
AcceptedCmp4           0
AcceptedCmp5           0
AcceptedCmp1           0
AcceptedCmp2           0
Complain               0
Z_CostContact          0
Z_Revenue              0
Response               0
dtype: int64

In [8]:
# generate age column, drop year_birth, handle data type issues

data['age'] = [2024 - x for x in data['Year_Birth']] # create age
data.drop(columns=['Year_Birth'], inplace=True) # drop year of birth


In [9]:
# converting columns to integer
data = data[data['Income'] != ' ']
data.reset_index(drop = True, inplace = True)

data = data.astype({
    'Income': 'int'
})

In [10]:
# inspecting the categorical columns

data['Education'].unique()
data['Marital_Status'].value_counts()

Marital_Status
Married     857
Together    573
Single      471
Divorced    232
Widow        76
Alone         3
Absurd        2
YOLO          2
Name: count, dtype: int64

In [11]:
data['Marital_Status'] = ['single' if x in ['YOLO','Absurd','Alone','Single'] else x for x in data['Marital_Status']]

In [22]:
cat_cols = ['Marital_Status','Education']
encoder = LabelEncoder()
for cols in cat_cols:
    data[f'{cols}_nums'] = encoder.fit_transform(data[cols])

In [13]:
data.head()

,Education,Marital_Status,Income,Kidhome,Teenhome,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,...,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response,age,Marital_Status_nums,Education_nums
0,Graduation,single,58138,0,0,58,635,88,546,172,...,0,0,0,0,3,11,1,67,4,2
1,Graduation,single,46344,1,1,38,11,1,6,2,...,0,0,0,0,3,11,0,70,4,2
2,Graduation,Together,71613,0,0,26,426,49,127,111,...,0,0,0,0,3,11,0,59,2,2
3,Graduation,Together,26646,1,0,26,11,4,20,10,...,0,0,0,0,3,11,0,40,2,2
4,PhD,Married,58293,1,0,94,173,43,118,46,...,0,0,0,0,3,11,0,43,1,4


In [14]:
final_data = data.drop(columns = cat_cols)

In [24]:
# modelling the data

scaler = StandardScaler()
scaled_df = scaler.fit_transform(final_data)
final_data = pd.DataFrame(data=scaled_df, columns=final_data.columns)
pca = PCA(n_components= 3)
columns = ['col1','col2','col3']
scaled_df = pca.fit_transform(final_data)
scaled_df = pd.DataFrame(data=scaled_df, columns=columns)

In [27]:
# clustering 

kmeans = k_means(X = scaled_df, n_clusters=4,n_init= 'auto')

In [28]:
predictions = kmeans[1]

In [33]:
px.scatter_3d(data_frame=scaled_df, x = 'col1', y = 'col2', z = 'col3',
              color= predictions)

In [ ]:
### Analysiscs of the graph
Overall, a 3D scatter plot provides a comprehensive visualization of the
relationship between multiple variables and can help in understanding patterns 
and relationships in your data, as well as assessing the performance of predictive models.





In [30]:
# clustering 

kmeans = k_means(X = scaled_df, n_clusters=4,n_init= 'auto')

In [31]:
predictions = kmeans[1]

In [32]:
data['clusters'] = ['cl1' if x == 1 else 'cl2' if x == 2 else
                    'cl3' if x == 3 else 'cl4' for x in predictions]
px.histogram(data_frame=data, x = 'clusters', y = 'Income',
             histfunc='avg', color = 'Education', barmode='group')


In [ ]:
px.histogram(data_frame=data, x = 'clusters', y = 'Kidhome',
             histfunc='avg', color = 'clusters')


### analytics of the graph
In the provided code snippet, you are creating a histogram using Plotly Express, 
where you are plotting the average values of the variable `Kidhome` across different 
clusters specified in the column `clusters`. Let's analyze the potential insights from this graph:

1. **X-axis (clusters):** This axis represents the different clusters identified in your data. 
    Each bar in the histogram corresponds to a specific cluster.

2. **Y-axis (average value of Kidhome):** This axis represents the average value of the variable `Kidhome` within each cluster.

3. **Color:** The bars in the histogram are likely to be colored according to the different clusters, 
    allowing for visual differentiation.

With this setup, the histogram provides insights into the distribution of the average values of `Kidhome` across the clusters.
Here's what you could potentially infer from this graph:

- **Comparison of Average Kidhome Values:** You can compare the average values of `Kidhome` 
    across different clusters. This analysis can help identify clusters with higher or lower average numbers of children at home.

- **Segmentation Analysis:** By observing how the average values of `Kidhome` vary across clusters, 
    you can potentially identify distinct segments within your customer base. For example, clusters with higher average 
    values of `Kidhome` may represent households with more children, while clusters with lower average values 
    may represent households with fewer children.

- **Targeting Strategies:** Insights from this analysis can inform targeted marketing strategies.
    For instance, if certain clusters have a higher average number of children at home,
    you might tailor marketing campaigns to appeal to families with children in those clusters.

Overall, this histogram provides a visual representation of the average values of `Kidhome` across different clusters,
enabling you to derive insights into the distribution of this variable within your dataset and potentially 
informing marketing strategies or segmentation approaches.


# Recomendation
Based on the histogram  generated, which displays the average values of the variable `Kidhome` across different clusters, here are some potential recommendations:

1. **Targeted Marketing Campaigns:** Identify clusters with higher average values of `Kidhome`, indicating households with more children. Tailor marketing campaigns to appeal to the needs and preferences of families with children in these clusters. This could include promotions for family-friendly products or services.

2. **Product Development:** Use insights from clusters with varying average values of `Kidhome` to inform product development or customization. Develop products or services that cater to the specific needs of households with different numbers of children.

3. **Customer Segmentation:** Utilize the clustering analysis to segment your customer base based on the average number of children at home. This segmentation can help personalize marketing strategies, communication channels, and product offerings for each segment.

4. **Customer Retention:** Identify clusters with lower average values of `Kidhome` and consider strategies to retain these customers. Understanding the unique needs of households without children can help tailor retention efforts and maintain customer satisfaction.

5. **Market Expansion:** Explore opportunities to expand into markets or target demographic segments with higher concentrations of households with children. This could involve opening new locations, launching specific marketing campaigns, or partnering with relevant organizations or platforms frequented by families.

6. **Data Analysis Refinement:** Continuously refine your data analysis techniques to uncover deeper insights into customer behavior and preferences. Consider incorporating additional variables or refining clustering algorithms to enhance the accuracy and granularity of your segmentation.

By leveraging the insights gained from the histogram and implementing these recommendations, you can effectively target and engage with different segments of your customer base, ultimately driving business growth and customer satisfaction.

In [ ]:
px.histogram(data_frame=data, x = 'clusters', y = 'MntWines',
             histfunc='avg', color = 'clusters')

### analytics of the graph
The histogram  created using Plotly Express shows the average spending on wines (`MntWines`) across different clusters. Let's analyze the insights this graph might offer:

1. **Cluster Comparison:** By comparing the average spending on wines across clusters, you can identify clusters that exhibit different purchasing behaviors. Clusters with higher average spending may indicate segments of customers who are more interested in premium or high-quality wines, while clusters with lower average spending may represent segments that are more price-conscious or less interested in wine products.

2. **Segmentation Insights:** The histogram helps in understanding how customers are distributed across different spending levels within each cluster. It allows you to identify clusters where there might be significant variations in wine spending behavior among customers, enabling more targeted marketing strategies.

3. **Identifying High-Value Segments:** Clusters with above-average spending on wines could represent high-value segments that offer opportunities for increased revenue and profitability. These segments may warrant special attention in terms of marketing efforts, product offerings, and customer engagement strategies.

4. **Market Opportunities:** Analyzing clusters with lower-than-average spending on wines can help identify potential growth opportunities. It allows you to understand which segments may be underrepresented in your current customer base and develop strategies to attract and retain customers from these segments.

5. **Customer Behavior Trends:** The histogram provides insights into overall trends in wine spending behavior across clusters. Understanding these trends can help in forecasting future demand, optimizing inventory management, and developing pricing strategies that align with customer preferences.

6. **Comparison with Business Objectives:** Analyze how the observed wine spending patterns align with your business objectives and goals. For example, if one of your objectives is to increase sales of premium wines, you can assess which clusters show the most potential for achieving this objective and prioritize resource allocation accordingly.

Overall, the histogram offers a visual representation of the average spending on wines across different clusters, facilitating the identification of customer segments with distinct purchasing behaviors and providing valuable insights for strategic decision-making in marketing, product development, and customer relationship management.

In [ ]:
px.histogram(data_frame=data, x = 'clusters', y ='Kidhome',
             histfunc='avg', color = 'clusters')

# cl3 have more kid while cl1 as the least kid

In [ ]:
px.histogram(data_frame=data, x = 'clusters', y ='MntFruits',
             histfunc='avg', color = 'clusters')

# cl2 love fruit while cl3 hate fruit

In [ ]:
px.histogram(data_frame=data, x = 'clusters', y ='Marital_Status',
             histfunc='count', color = 'Marital_Status')

# cl3 have the most married,widow,single and divorced status

In [ ]:
px.histogram(data_frame=data, x = 'clusters', y ='Teenhome',
             histfunc='avg', color = 'clusters')

# cl4 have the most teen children

In [ ]:
px.histogram(data_frame=data, x = 'clusters', y ='Income',
             histfunc='avg', color = 'clusters')            

# cl1 is the richest while cl3 is poor 

In [ ]:
prx.histogram(data_frame=data, x = 'clusters', y ='MntMeatProducts',
            histfunc='avg', color = 'clusters')

# cl1 buy more products than the others whlie cl3 bought the least

In [ ]:
px.histogram(data_frame=data, x = 'clusters', y ='Education',
             histfunc='count', color = 'Education')

# cl3 have the most graduate while cl4 has the most PhD while cl3 has the most masters also cl3 have the most basic and 2n cycle

#  while cl1 have less education 

In [ ]:
px.histogram(data_frame=data, x = 'clusters', y ='Complain',
             histfunc='count', color = 'clusters')

# cl3 complain tooo much

In [ ]:
px.histogram(data_frame=data, x = 'clusters', y ='Complain',
             histfunc='count', color = 'clusters')
